In [1]:
from matplotlib.backends.backend_pdf import PdfPages
import argparse
import pathlib
import importlib

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import json
import uproot
import sys
import awkward as ak
sys.path.append("../../scripts/")

from pathlib import Path
import common_functions as au
from baseline_chi2pid import passes_kplus_chi2pid_cut


In [2]:
import awkward as ak
import math

def compute_RICH_contam(df, pid=None):

    temp = df

    if pid is None:
        # Total contamination
        a = ak.sum(temp["rich_best_PID"] != 321)
    else:
        # Contribution from one particle species
        a = ak.sum(temp["rich_best_PID"] == pid)

    b = len(temp)

    r = 0.0
    rErr = 99.0

    if b != 0:
        r = float(a) / b

    if a != 0:
        rErr = r * math.sqrt((1 / float(a)) + (1 / b))

    return r, rErr

In [3]:
cols = ["pid", "p", "theta", "chi2pid", "rich_RQ", "vz", "bdt_pass", "rich_best_PID", "rich_RQ", "rich_best_ntot"]
kinematics =["Mx_eKX","Mx_epiX","Mx_epX", "Q2", "W", "y"]

for kin in kinematics:
    cols.append(kin)
    
#df_dat = uproot.open("~/ML_Files/epkx_data/scored_larger/larger.root:PhysicsEvents").arrays(cols, library="pd")
df_dat = uproot.open("~/ML_Files/scored_data_v01/nSidis_005046.root:PhysicsEvents").arrays(cols, library="pd")



df_dat=df_dat[df_dat["rich_best_PID"]==321]

cols.append("mc_matching_pid")

df_mc = uproot.open("~/ML_Files/MC_scored/pid_training_v2.root:PhysicsEvents").arrays(cols, library="pd")


df_mc=df_mc[df_mc["rich_best_PID"]==321]
df_mc=au.apply_RICH_Quality_Cuts(df_mc)
df_dat=au.apply_RICH_Quality_Cuts(df_dat)
df_mc=au.apply_Sidis_Cuts(df_mc)
df_dat=au.apply_Sidis_Cuts(df_dat)
outDir="../../figures/Data_Application/EB_Kaon/"

allPlots=[]

In [4]:
tStart = 5
tEnd = 11.25
tBinNum = 5

pStart = 2.5
pEnd = 5
pBinNum = 10

pEdges = np.linspace(pStart, pEnd, pBinNum + 1)



In [5]:
pBins_mc = au.makeBins(df_mc, "p", binEdges=pEdges)
pBins_dat = au.makeBins(df_dat, "p", binEdges=pEdges)
vals_dat=[]
errs_dat=[]

for pbin in pBins_dat:
    subset=pbin[pbin["bdt_pass"]]
    va,er=compute_RICH_contam(subset)
    vals_dat.append(va)
    errs_dat.append(er)

vals_mc=[]
errs_mc=[]

for pbin in pBins_mc:
    subset=pbin[pbin["bdt_pass"]]
    va,er=au.compute_contamination_ak(subset)
    vals_mc.append(va)
    errs_mc.append(er)


    

In [6]:
import matplotlib.pyplot as plt
import numpy as np

outdir="../../figures/Data_Application/"



# Calculate bin centers
p_centers = (pEdges[:-1] + pEdges[1:]) / 2

# DATA plot
plt.figure(figsize=(8,6))

plt.errorbar(
    p_centers,
    vals_dat,
    yerr=errs_dat,
    fmt='o',
    capsize=3
)

plt.xlabel(r"$p$ (GeV/c)")
plt.ylabel("Contamination")
plt.title("Contamination vs Momentum (RICH)")
plt.grid(False)

plt.savefig(outdir+"contamination_data.png", dpi=150, bbox_inches="tight")
plt.close()


# MC plot
plt.figure(figsize=(8,6))

plt.errorbar(
    p_centers,
    vals_mc,
    yerr=errs_mc,
    fmt='o',
    capsize=3
)

plt.xlabel(r"$p$ (GeV/c)")
plt.ylabel("Contamination")
plt.title("Contamination vs Momentum (MC)")
plt.grid(True)

plt.savefig(outdir+"contamination_mc.png", dpi=150, bbox_inches="tight")
plt.close()


# DATA + MC overlay
plt.figure(figsize=(8,6))

plt.errorbar(
    p_centers,
    vals_dat,
    yerr=errs_dat,
    fmt='o',
    capsize=3,
    label="DATA"
)

plt.errorbar(
    p_centers,
    vals_mc,
    yerr=errs_mc,
    fmt='o',
    capsize=3,
    label="MC"
)

plt.xlabel(r"$p$ (GeV/c)")
plt.ylabel("Contamination")
plt.title("Contamination vs Momentum")
plt.legend()
plt.grid(True)




plt.savefig(outdir+"both_contamination.png", dpi=150, bbox_inches="tight")
plt.close()